## Συμπέρασμα

Σε αυτό το notebook:

- επιθεωρήσαμε τα markdown outputs του parsing
- εντοπίσαμε κοινά extraction artifacts
- εφαρμόσαμε βασικό cleaning
- αποθηκεύσαμε cleaned markdown αρχεία
- δημιουργήσαμε cleaning manifest

Το επόμενο notebook θα είναι το `05_chunking.ipynb`.

In [27]:
# Uncomment if needed
# !pip install -q langchain-text-splitters pyarrow tqdm

In [28]:
from pathlib import Path
import json
import warnings
import re

import pandas as pd
from tqdm.auto import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [29]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

IN_KAGGLE = Path("/kaggle/working").exists()
print("IN_KAGGLE:", IN_KAGGLE)

IN_KAGGLE: False


In [30]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = BASE_DIR / "configs"
OUTPUTS_DIR = BASE_DIR / "outputs"

CLEANED_DIR = INTERIM_DIR / "cleaned"
CHUNKS_DIR = PROCESSED_DIR / "chunks"

PARSE_MANIFEST_PATH = INTERIM_DIR / "docling_parse_manifest.csv"
CLEANING_MANIFEST_PATH = INTERIM_DIR / "markdown_cleaning_manifest.csv"

CHUNKS_CSV_PATH = CHUNKS_DIR / "financebench_chunks.csv"
CHUNKS_PARQUET_PATH = CHUNKS_DIR / "financebench_chunks.parquet"
CHUNKING_MANIFEST_PATH = CHUNKS_DIR / "chunking_manifest.csv"
CHUNKING_STATS_PATH = CHUNKS_DIR / "chunking_stats.json"

for p in [CHUNKS_DIR, OUTPUTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("CLEANED_DIR:", CLEANED_DIR)
print("CHUNKS_DIR:", CHUNKS_DIR)

CLEANED_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned
CHUNKS_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks


In [31]:
parse_manifest_df = pd.read_csv(PARSE_MANIFEST_PATH)
cleaning_manifest_df = pd.read_csv(CLEANING_MANIFEST_PATH)

print("parse_manifest_df:", parse_manifest_df.shape)
print("cleaning_manifest_df:", cleaning_manifest_df.shape)

parse_manifest_df: (84, 15)
cleaning_manifest_df: (84, 15)


In [32]:
successful_cleaning_df = cleaning_manifest_df[cleaning_manifest_df["status"] == "success"].copy().reset_index(drop=True)

print("Successful cleaned files:", len(successful_cleaning_df))
successful_cleaning_df.head(2)

Successful cleaned files: 84


,markdown_filename,markdown_path,cleaned_markdown_path,status,error_message,removed_checkbox_lines,removed_standalone_page_numbers,removed_low_lines,collapsed_blank_line_groups,trimmed_lines,original_chars,cleaned_chars,char_delta,original_lines,cleaned_lines
0,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\markdown\3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,success,NaN,0,7,1,6,0,836595,627839,-208756,4726,4710
1,3M_2022_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\markdown\3M_2022_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2022_10K.md,success,NaN,0,0,0,0,0,1183135,975344,-207791,6570,6570


In [33]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
MIN_CHUNK_CHARS = 200

chunking_config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "min_chunk_chars": MIN_CHUNK_CHARS
}

chunking_config

{'chunk_size': 1000, 'chunk_overlap': 150, 'min_chunk_chars': 200}

In [34]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n# ",
        "\n## ",
        "\n### ",
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],
    keep_separator=True
)

print(text_splitter)

In [35]:
def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def normalize_whitespace(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def estimate_tokens(text: str) -> int:
    # rough approximation for analysis only
    return max(1, len(text) // 4)

def safe_doc_id_from_filename(filename: str) -> str:
    return Path(filename).stem

In [36]:
doc_inventory = successful_cleaning_df.copy()

doc_inventory["cleaned_markdown_exists"] = doc_inventory["cleaned_markdown_path"].apply(lambda x: Path(x).exists())
doc_inventory["doc_id"] = doc_inventory["markdown_filename"].apply(safe_doc_id_from_filename)

print("Inventory shape:", doc_inventory.shape)
doc_inventory.head(2)

Inventory shape: (84, 17)


,markdown_filename,markdown_path,cleaned_markdown_path,status,error_message,removed_checkbox_lines,removed_standalone_page_numbers,removed_low_lines,collapsed_blank_line_groups,trimmed_lines,original_chars,cleaned_chars,char_delta,original_lines,cleaned_lines,cleaned_markdown_exists,doc_id
0,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\markdown\3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,success,NaN,0,7,1,6,0,836595,627839,-208756,4726,4710,True,3M_2018_10K
1,3M_2022_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\markdown\3M_2022_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2022_10K.md,success,NaN,0,0,0,0,0,1183135,975344,-207791,6570,6570,True,3M_2022_10K


In [37]:
doc_inventory = doc_inventory[doc_inventory["cleaned_markdown_exists"]].copy().reset_index(drop=True)

print("Docs available for chunking:", len(doc_inventory))
doc_inventory.head(2)

Docs available for chunking: 84


,markdown_filename,markdown_path,cleaned_markdown_path,status,error_message,removed_checkbox_lines,removed_standalone_page_numbers,removed_low_lines,collapsed_blank_line_groups,trimmed_lines,original_chars,cleaned_chars,char_delta,original_lines,cleaned_lines,cleaned_markdown_exists,doc_id
0,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\markdown\3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,success,NaN,0,7,1,6,0,836595,627839,-208756,4726,4710,True,3M_2018_10K
1,3M_2022_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\markdown\3M_2022_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2022_10K.md,success,NaN,0,0,0,0,0,1183135,975344,-207791,6570,6570,True,3M_2022_10K


In [38]:
sample_path = Path(doc_inventory.loc[0, "cleaned_markdown_path"]) if len(doc_inventory) > 0 else None

if sample_path:
    sample_text = read_text(sample_path)
    print("Sample file:", sample_path.name)
    print(sample_text[:3000])
else:
    print("No cleaned markdown files found.")

Sample file: 3M_2018_10K.md
## UNITED STATES SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

## FORM 10-K

☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2018

Commission file number 1-3285

## 3M COMPANY

State of Incorporation:

Delaware

I.R.S. Employer Identification No.

41-0417775

Principal executive offices:

3M Center, St. Paul, Minnesota 55144

Telephone number:

(651) 733-1110

## SECURITIES REGISTERED PURSUANT TO SECTION 12(b) OF THE ACT:

Name of each exchange

## Title of each class

on which registered

Common Stock, Par Value $.01 Per Share

1.500% Notes due 2026

Floating Rate Notes due 2020

0.375% Notes due 2022

0.950% Notes due 2023

1.750% Notes due 2030

1.500% Notes due 2031

New York Stock Exchange, Inc.

Chicago Stock Exchange, Inc.

New York Stock Exchange, Inc.

New York Stock Exchange, Inc.

New York Stock Exchange, Inc.

New York Stock Exchange, Inc.

New York Stock 

In [39]:
if sample_path:
    sample_text = normalize_whitespace(read_text(sample_path))
    sample_chunks = text_splitter.split_text(sample_text)

    print("Number of sample chunks:", len(sample_chunks))
    print("\nFirst chunk preview:\n")
    print(sample_chunks[0][:2000])
else:
    print("No sample available.")

Number of sample chunks: 1038

First chunk preview:

## UNITED STATES SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

## FORM 10-K

☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2018

Commission file number 1-3285

## 3M COMPANY

State of Incorporation:

Delaware

I.R.S. Employer Identification No.

41-0417775

Principal executive offices:

3M Center, St. Paul, Minnesota 55144

Telephone number:

(651) 733-1110

## SECURITIES REGISTERED PURSUANT TO SECTION 12(b) OF THE ACT:

Name of each exchange


In [40]:
def chunk_document(doc_id: str, markdown_filename: str, markdown_path: Path, company=None, doc_type=None, doc_period=None):
    raw_text = read_text(markdown_path)
    normalized_text = normalize_whitespace(raw_text)

    split_chunks = text_splitter.split_text(normalized_text)

    records = []
    for i, chunk_text in enumerate(split_chunks):
        chunk_text = chunk_text.strip()

        if len(chunk_text) < MIN_CHUNK_CHARS:
            continue

        chunk_id = f"{doc_id}_chunk_{i:04d}"

        record = {
            "chunk_id": chunk_id,
            "doc_id": doc_id,
            "doc_name": doc_id,
            "markdown_filename": markdown_filename,
            "source_path": str(markdown_path),
            "company": company,
            "doc_type": doc_type,
            "doc_period": doc_period,
            "chunk_index": i,
            "chunk_text": chunk_text,
            "char_count": len(chunk_text),
            "token_estimate": estimate_tokens(chunk_text),
            "starts_with_heading": bool(re.match(r"^\s*#+\s+", chunk_text)),
            "contains_table_pipe": "|" in chunk_text,
        }

        records.append(record)

    return records

In [41]:
all_chunk_records = []
chunking_manifest_records = []

for _, row in tqdm(doc_inventory.iterrows(), total=len(doc_inventory), desc="Chunking documents"):
    markdown_filename = row["markdown_filename"]
    markdown_path = Path(row["cleaned_markdown_path"])
    doc_id = row["doc_id"]

    record = {
        "doc_id": doc_id,
        "markdown_filename": markdown_filename,
        "cleaned_markdown_path": str(markdown_path),
        "status": None,
        "error_message": None,
        "n_chunks": 0,
        "total_chars": None,
    }

    try:
        raw_text = read_text(markdown_path)
        chunk_records = chunk_document(
            doc_id=doc_id,
            markdown_filename=markdown_filename,
            markdown_path=markdown_path,
            company=row.get("company"),
            doc_type=row.get("doc_type"),
            doc_period=row.get("doc_period"),
        )

        all_chunk_records.extend(chunk_records)

        record["status"] = "success"
        record["n_chunks"] = len(chunk_records)
        record["total_chars"] = len(raw_text)

    except Exception as e:
        record["status"] = "error"
        record["error_message"] = str(e)

    chunking_manifest_records.append(record)

chunks_df = pd.DataFrame(all_chunk_records)
chunking_manifest_df = pd.DataFrame(chunking_manifest_records)

print("chunks_df shape:", chunks_df.shape)
print("chunking_manifest_df shape:", chunking_manifest_df.shape)

Chunking documents: 100%|██████████| 84/84 [00:02<00:00, 34.59it/s]


chunks_df shape: (60835, 14)
chunking_manifest_df shape: (84, 7)


In [42]:
chunking_manifest_df.head()

,doc_id,markdown_filename,cleaned_markdown_path,status,error_message,n_chunks,total_chars
0,3M_2018_10K,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,success,None,913,627839
1,3M_2022_10K,3M_2022_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2022_10K.md,success,None,1377,975344
2,3M_2023Q2_10Q,3M_2023Q2_10Q.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2023Q2_10Q.md,success,None,545,389071
3,ACTIVISIONBLIZZARD_2019_10K,ACTIVISIONBLIZZARD_2019_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\ACTIVISIONBLIZZARD_2019_10K.md,success,None,955,660448
4,ADOBE_2015_10K,ADOBE_2015_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\ADOBE_2015_10K.md,success,None,722,506487


In [43]:
chunks_df.head(3)

,chunk_id,doc_id,doc_name,markdown_filename,source_path,company,doc_type,doc_period,chunk_index,chunk_text,char_count,token_estimate,starts_with_heading,contains_table_pipe
0,3M_2018_10K_chunk_0000,3M_2018_10K,3M_2018_10K,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,None,None,None,0,"## UNITED STATES SECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\n## FORM 10-K\n\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fisc...",540,135,True,False
1,3M_2018_10K_chunk_0001,3M_2018_10K,3M_2018_10K,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,None,None,None,1,"## Title of each class\n\non which registered\n\nCommon Stock, Par Value $.01 Per Share\n\n1.500% Notes due 2026\n\nFloating Rate Notes due 2020\n\n0.375% Notes due 2022\n\n0.950% Notes due 2023\n...",914,228,True,False
2,3M_2018_10K_chunk_0002,3M_2018_10K,3M_2018_10K,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,None,None,None,2,- [ ] ☐\n\n- [x] No ☒\n\nIndicate by check mark whether the Registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the prece...,671,167,False,False


In [44]:
summary = {
    "n_documents": int(chunking_manifest_df["doc_id"].nunique()) if not chunking_manifest_df.empty else 0,
    "successful_documents": int((chunking_manifest_df["status"] == "success").sum()) if not chunking_manifest_df.empty else 0,
    "failed_documents": int((chunking_manifest_df["status"] == "error").sum()) if not chunking_manifest_df.empty else 0,
    "total_chunks": len(chunks_df),
    "avg_chunks_per_document": float(chunks_df.groupby("doc_id").size().mean()) if not chunks_df.empty else 0,
    "avg_chunk_chars": float(chunks_df["char_count"].mean()) if not chunks_df.empty else 0,
    "avg_chunk_token_estimate": float(chunks_df["token_estimate"].mean()) if not chunks_df.empty else 0,
}

pd.DataFrame([summary])

,n_documents,successful_documents,failed_documents,total_chunks,avg_chunks_per_document,avg_chunk_chars,avg_chunk_token_estimate
0,84,84,0,60835,724.22619,705.251352,175.94135


In [45]:
chunks_df["char_count"].describe()

count    60835.000000
mean       705.251352
std        230.743619
min        200.000000
25%        527.000000
50%        756.000000
75%        910.000000
max       1000.000000
Name: char_count, dtype: float64

In [46]:
short_chunks_df = chunks_df[chunks_df["char_count"] < MIN_CHUNK_CHARS].copy()

print("Short chunks:", len(short_chunks_df))
short_chunks_df.head()

Short chunks: 0


,chunk_id,doc_id,doc_name,markdown_filename,source_path,company,doc_type,doc_period,chunk_index,chunk_text,char_count,token_estimate,starts_with_heading,contains_table_pipe


In [47]:
per_doc_chunk_counts = (
    chunks_df.groupby("doc_id")
    .size()
    .reset_index(name="n_chunks")
    .sort_values("n_chunks", ascending=False)
)

per_doc_chunk_counts.head(15)

,doc_id,n_chunks
50,JPMORGAN_2022_10K,2354
37,CVSHEALTH_2018_10K,2039
71,PEPSICO_2022_10K,1753
70,PEPSICO_2021_10K,1632
21,AMERICANWATERWORKS_2022_10K,1501
81,WALMART_2018_10K,1387
1,3M_2022_10K,1377
8,AES_2022_10K,1359
38,CVSHEALTH_2022_10K,1280
18,AMERICANEXPRESS_2022_10K,1273


In [48]:
sample_doc_id = chunks_df["doc_id"].iloc[0] if not chunks_df.empty else None

if sample_doc_id:
    sample_doc_chunks = chunks_df[chunks_df["doc_id"] == sample_doc_id].copy()
    print("Sample doc_id:", sample_doc_id)
    print("Number of chunks:", len(sample_doc_chunks))

    for _, row in sample_doc_chunks.head(3).iterrows():
        print("\n" + "=" * 100)
        print("chunk_id:", row["chunk_id"])
        print("chunk_index:", row["chunk_index"])
        print("char_count:", row["char_count"])
        print("-" * 100)
        print(row["chunk_text"][:2000])
else:
    print("No chunks available.")

Sample doc_id: 3M_2018_10K
Number of chunks: 913

chunk_id: 3M_2018_10K_chunk_0000
chunk_index: 0
char_count: 540
----------------------------------------------------------------------------------------------------
## UNITED STATES SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

## FORM 10-K

☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2018

Commission file number 1-3285

## 3M COMPANY

State of Incorporation:

Delaware

I.R.S. Employer Identification No.

41-0417775

Principal executive offices:

3M Center, St. Paul, Minnesota 55144

Telephone number:

(651) 733-1110

## SECURITIES REGISTERED PURSUANT TO SECTION 12(b) OF THE ACT:

Name of each exchange

chunk_id: 3M_2018_10K_chunk_0001
chunk_index: 1
char_count: 914
----------------------------------------------------------------------------------------------------
## Title of each class

on which registered

Common Stock, Par Value $.01 Per 

In [49]:
chunks_df.to_csv(CHUNKS_CSV_PATH, index=False)
print("Saved CSV to:", CHUNKS_CSV_PATH)

try:
    chunks_df.to_parquet(CHUNKS_PARQUET_PATH, index=False)
    print("Saved Parquet to:", CHUNKS_PARQUET_PATH)
except ImportError as e:
    print("Parquet save skipped.")
    print("Reason:", e)

Saved CSV to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks\financebench_chunks.csv
Saved Parquet to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks\financebench_chunks.parquet


In [50]:
chunking_manifest_df.to_csv(CHUNKING_MANIFEST_PATH, index=False)
print("Saved manifest to:", CHUNKING_MANIFEST_PATH)

Saved manifest to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks\chunking_manifest.csv


In [51]:
with open(CHUNKING_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved stats JSON to:", CHUNKING_STATS_PATH)

Saved stats JSON to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks\chunking_stats.json


In [52]:
output_summary = pd.DataFrame([{
    "chunks_csv_path": str(CHUNKS_CSV_PATH),
    "chunks_parquet_path": str(CHUNKS_PARQUET_PATH),
    "chunking_manifest_path": str(CHUNKING_MANIFEST_PATH),
    "chunking_stats_path": str(CHUNKING_STATS_PATH),
    "n_chunks": len(chunks_df),
    "n_docs_chunked": chunking_manifest_df["doc_id"].nunique() if not chunking_manifest_df.empty else 0
}])

output_summary

,chunks_csv_path,chunks_parquet_path,chunking_manifest_path,chunking_stats_path,n_chunks,n_docs_chunked
0,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks\financebench_chunks.csv,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks\financebench_chunks.parquet,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks\chunking_manifest.csv,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks\chunking_stats.json,60835,84


## Συμπέρασμα

Σε αυτό το notebook:

- φορτώσαμε τα cleaned markdown documents
- εφαρμόσαμε recursive chunking
- κρατήσαμε metadata ανά chunk
- αποθηκεύσαμε chunks και chunking manifest
- δημιουργήσαμε βασικά στατιστικά

Το επόμενο notebook θα είναι το `06_embeddings_and_vectorstore.ipynb`.